← [Previous: Chapter 1: Getting Started](https://colab.research.google.com/github/najaeda/naja/blob/main/tutorials/notebooks/01_getting_started.ipynb) · 🔗 [Back to Table of Contents](https://github.com/najaeda/naja/tree/main/tutorials)
# Chapter 2: Opening a Design with Primitives Described in Liberty Format

In this chapter, we’ll explore how to work with a [design](https://github.com/chipsalliance/rocket-chip) generated using the **Rocket Chip** generator.

While detailed information about the design can be found in [this technical report](https://www2.eecs.berkeley.edu/Pubs/TechRpts/2016/EECS-2016-17.html), our goal here is not to study its functionality.  
Instead, we’ll focus on how to load a synthesized design that makes use of a standard-cell library defined in the **Liberty format**.

The **Liberty format** is an industry-standard way to describe the behavior, timing, and characteristics of library cells in a specific technology node.

For this tutorial, we’ll use the open-source [Nangate](https://github.com/oscc-ip/nangate) library as our standard-cell library.

---

Let's begin by setting up the environment.

In [ ]:
import os
!wget https://github.com/najaeda/naja/archive/refs/heads/main.zip
!unzip main.zip
os.environ['NAJAEDA_TUTORIALS_BENCHMARKS'] = 'naja-main/tutorials/benchmarks'
!pip install 'najaeda>=0.3.4'

## Loading the liberty library and the design

After importing **najaeda** package, we construct a list of the needed **liberty** files, we load them and then we load the verilog design. 

In [ ]:
import os
from pathlib import Path
from najaeda import netlist

# In Colab: set by the setup cell above.
# In CI: passed as NAJAEDA_TUTORIALS_BENCHMARKS env var.
# Locally: defaults to ../benchmarks relative to this notebook.
_benchmarks = os.environ.get(
    'NAJAEDA_TUTORIALS_BENCHMARKS',
    str(Path().resolve().parent / 'benchmarks')
)
liberty_files = [
    f'{_benchmarks}/liberty/NangateOpenCellLibrary_typical.lib',
    f'{_benchmarks}/liberty/fakeram45_64x32.lib'
]
netlist.load_liberty(liberty_files)
top = netlist.load_verilog(f'{_benchmarks}/verilog/tinyrocket/tinyrocket.v')
print(f"Design {top.get_name()} loaded")


## Traversing the Design Hierarchy

The most direct approach is to write a simple recursive function that walks the instance tree. This is useful for understanding the structure, but for production use `najaeda` provides the `instance_visitor` API which handles traversal efficiently and supports pruning, early exit, and argument passing.

In [ ]:
def explore_design(instance):
    print(f"{instance.get_name()} with model: {instance.get_model_name()}")
    for ins in instance.get_child_instances():
        explore_design(ins)

explore_design(top)

## Using `instance_visitor`

`instance_visitor` replaces the manual recursive traversal above. It visits every instance in the hierarchy and calls your callback for each one.

In [ ]:
from najaeda import instance_visitor

def print_instance(instance):
    print(f"{instance.get_name()} with model: {instance.get_model_name()}")

visitor_config = instance_visitor.VisitorConfig(callback=print_instance)
instance_visitor.visit(top, visitor_config)

We can modify the `print_instance` method to print only non-primitive instances, effectively showing only the user-defined hierarchy.

In [ ]:
def print_non_primitive(instance):
    if not instance.is_primitive():
        print(f"{instance} with model: {instance.get_model_name()}")

visitor_config = instance_visitor.VisitorConfig(callback=print_non_primitive)
instance_visitor.visit(top, visitor_config)

It's also possible to pass additional arguments to the callback function used with the `instance_visitor` class.

As an example, let's implement a function that counts all primitive and non primitive instances across the entire design.  
To do this, we'll use a Python dictionary to pass a mutable counter to the callback.

In [ ]:
instance_counts = {"primitives": 0, "non_primitives": 0}
def count_instances(instance, instance_counts):
    if instance.is_primitive():
        instance_counts["primitives"] += 1
    else:
        instance_counts["non_primitives"] += 1

visitor_config = instance_visitor.VisitorConfig(callback=count_instances, args=(instance_counts,))
instance_visitor.visit(top, visitor_config)
print(f"In design {top.get_name()}:")
print(f"Total number of primitive instances: {instance_counts['primitives']}")
print(f"Total number of non-primitive instances: {instance_counts['non_primitives']}")

### Extracting Source Information

Another practical use case is extracting source location data from the netlist, when available.

In the case of `yosys`, for example, this information is stored in an attribute named `src`, with values like:

```text
src = "/designs/src/tinyRocket/freechips.rocketchip.system.TinyConfig.v:134838.3-134904.6"
```

To begin, we’ll enhance our instance traversal to print attributes — but only for non-primitive instances, to keep the output concise and readable.

In [ ]:
def print_attributes(instance):
    if not instance.is_primitive():
        print(f"{instance} with model: {instance.get_model_name()}")
        attrs = instance.get_attributes()
        if attrs:
            print("Attributes:")
            for attr in attrs:
                print(f"  {attr.get_name()}: {attr.get_value()}")
        else:
            print("No attributes.")

visitor_config = instance_visitor.VisitorConfig(callback=print_attributes)
instance_visitor.visit(top, visitor_config)

We’ll now implement a function to decode the src attribute into a more usable format.

In [ ]:
import re

def parse_src_location(src_str):
    match = re.match(r"(.+):(\d+)\.(\d+)-(\d+)\.(\d+)", src_str)
    if match:
        path = match.group(1)
        start_line = int(match.group(2))
        start_col = int(match.group(3))
        end_line = int(match.group(4))
        end_col = int(match.group(5))
        return {
            "file": path,
            "start": (start_line, start_col),
            "end": (end_line, end_col),
        }
    return None

def extract_src_information(instance):
    if not instance.is_primitive():
        print(f"{instance} with model: {instance.get_model_name()}")
        attrs = instance.get_attributes()
        for attr in attrs:
            if "src" in attr.get_name():
                src_info = parse_src_location(attr.get_value())
                if src_info:
                    print(f"Source file: {src_info['file']}")
                    print(f"Defined from line {src_info['start'][0]}, column {src_info['start'][1]} "
                          f"to line {src_info['end'][0]}, column {src_info['end'][1]}")

visitor_config = instance_visitor.VisitorConfig(callback=extract_src_information)
instance_visitor.visit(top, visitor_config)

## Extracting Statistics and Using Pandas

With `najaeda`, it’s straightforward to extract design data and prepare it for analysis in popular frameworks such as [pandas](https://pandas.pydata.org/).

In fact, `najaeda` already includes built-in methods to generate this data in a format that’s ready for use.

As an example, let’s export statistics for each instance under `top` in `JSON` format.

In [ ]:
from najaeda import stats

stats.dump_instance_stats_json(top, 'top_stats.json')

Next, we’ll load the file into `Pandas` and display its contents.

In [ ]:
import pandas as pd

df = pd.read_json('top_stats.json')
pandas_data = pd.DataFrame(df.set_index("Name"))
plot = pandas_data.plot.bar(y=["terms", "nets", "instances"], stacked=True)

# Customize plot
plot.set_title("Design Statistics", fontsize=16, fontweight="bold")
plot.set_xlabel("Design Name", fontsize=12)
plot.set_ylabel("Count", fontsize=12)

plot_figure = plot.get_figure()
plot_figure.tight_layout()
plot_figure.savefig('design_stats.png')

This concludes **Chapter 2** of the **najaeda** tutorial.  
In the next chapter, we’ll dive into how to modify and manipulate the netlist structure.

➡️ [**Next Chapter → Editing a Netlist**](https://colab.research.google.com/github/najaeda/naja/blob/main/tutorials/notebooks/03_editing_a_netlist.ipynb)
